# MATH840 Final project — model family: regression and machine learning

Ridge regression, random forest, extra trees, histogram gradient boosting, LightGBM variants (also season- and recency-weighted, and on 3 years), Prophet, and analog days (nearest-neighbour weather).

Same data, same **20 rolling 48-hour windows** (one origin every midnight, 6–25 October), same scores as
`LUKIANETS_project_v2.ipynb`, judged on **day 2** (hours 25–48, the private leaderboard's hours). The last cell writes
`family_ml_bt.csv` and `family_ml_final.csv`; the main notebook merges them (Section 2.10) into one scoreboard, where the
combinations, the night rule and the final choice are made. Run it from top to bottom; the family notebooks are
independent of each other and can run at the same time. With `RESUME = True` (helpers cell) the models of an earlier
run are read back from the export and only new sections compute.

## 0. Setup

Download `train.csv` and `weather_forecast.csv` from Moodle. In Colab, the next cell asks you to upload
them. On your own machine, put them in the same folder as the notebook.

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120, "display.precision", 3)

SURNAME = "LUKIANETS"   # <-- for your submission file name

if not SURNAME.strip():
    raise ValueError("Set SURNAME before running the rest.")

In [2]:
NEEDED = ["train.csv", "weather_forecast.csv"]

if not all(Path(f).exists() for f in NEEDED):
    try:
        from google.colab import files
        files.upload()                       # choose both files at once
    except ImportError:
        raise FileNotFoundError(f"Put {NEEDED} next to this notebook.")

data = pd.read_csv("train.csv", parse_dates=["ds"])
weather_fc = pd.read_csv("weather_forecast.csv", parse_dates=["ds"])

H = 48                                       # hours to forecast
M = 24                                       # the benchmark's season: one day
FUTURE = weather_fc["ds"]

assert len(FUTURE) == H and FUTURE.iloc[0] == data["ds"].iloc[-1] + pd.Timedelta(hours=1)
print(f"history: {len(data)} hours, {data['ds'].min()} to {data['ds'].max()}")
print(f"you are forecasting: {FUTURE.iloc[0]} to {FUTURE.iloc[-1]}")
data.tail()

history: 33480 hours, 2023-01-01 01:00:00 to 2026-10-27 00:00:00
you are forecasting: 2026-10-27 01:00:00 to 2026-10-29 00:00:00


,ds,y,cloud_cover,temperature,wind_speed,pressure,humidity,daylight,air_density
33475,2026-10-26 20:00:00,42.5,1.0,9,3,763,87,0,1.259
33476,2026-10-26 21:00:00,76.7,1.0,9,4,763,87,0,1.259
33477,2026-10-26 22:00:00,65.8,1.0,7,3,763,100,0,1.268
33478,2026-10-26 23:00:00,69.9,1.0,7,3,763,100,0,1.268
33479,2026-10-27 00:00:00,69.9,1.0,7,3,763,100,0,1.268


### The data

| Column | Meaning |
|---|---|
| `ds` | the hour |
| `y` | electricity generated in that hour; this is what you forecast |
| `cloud_cover` | from 0 (clear) to 1 (overcast), in steps of 0.25 |
| `temperature` | °C |
| `wind_speed` | m/s |
| `pressure` | mm Hg |
| `humidity` | % |
| `daylight` | 1 while the sun is above the horizon |
| `air_density` | kg/m³ |

`train.csv` has the weather as it was. `weather_forecast.csv` has a forecast for the 48 hours you
predict, of the kind a weather service issues. Its errors grow from the first day to the second.

## 1. How you are scored

Your file gets three scores on the 48 hidden hours, and lower is better for each. RMSE measures the
point forecast in the units of `y`, and MAPE measures it in percent of the actual value. The Winkler
score measures the 95% interval: its width, plus 40 times ($2/\alpha$) the distance by which the actual
value falls outside it.

The benchmark is a seasonal naive forecast with a daily season. Each hour of the next two days gets the
value of the same hour on the last day you have. Its 95% interval is the standard one for seasonal naive,
$\hat y \pm 1.96\,\hat\sigma\sqrt{k+1}$, where $k$ is the number of full days into the horizon.

The project description on Moodle explains the points and the leaderboard.

In [3]:
def rmse(actual, forecast) -> float:
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(forecast)) ** 2)))


def mape(actual, forecast) -> float:
    actual, forecast = np.asarray(actual, float), np.asarray(forecast, float)
    return float(np.mean(np.abs((actual - forecast) / actual)) * 100)


def winkler(actual, lower, upper, alpha: float = 0.05) -> float:
    actual, lower, upper = (np.asarray(a, float) for a in (actual, lower, upper))
    below = np.clip(lower - actual, 0, None)
    above = np.clip(actual - upper, 0, None)
    return float(np.mean((upper - lower) + 2 / alpha * (below + above)))


def score(actual, forecast, lower, upper) -> dict:
    inside = (np.asarray(actual) >= lower) & (np.asarray(actual) <= upper)
    return {"RMSE": rmse(actual, forecast), "MAPE": mape(actual, forecast),
            "Winkler": winkler(actual, lower, upper), "coverage": float(inside.mean())}


def snaive(history: np.ndarray, h: int = H, m: int = M):
    # The benchmark: repeat the last season, with the seasonal-naive 95% interval.
    history = np.asarray(history, float)
    point = np.array([history[-m + (i % m)] for i in range(h)])
    sigma = np.std(history[m:] - history[:-m], ddof=1)
    k = np.arange(h) // m
    half = 1.96 * sigma * np.sqrt(k + 1)
    return point, np.clip(point - half, 0, None), point + half

### The benchmark on a validation window

You cannot score a forecast on the hidden hours, so the cell below holds back the last 48 hours of the
history, forecasts them from the hours before, and scores the result. This is a single window. You can
decide how much more validation you need.

In [4]:
y = data["y"].to_numpy(float)
past, valid = y[:-H], y[-H:]

bench = snaive(past)
results = pd.DataFrame({"seasonal naive": score(valid, *bench)}).T
results

,RMSE,MAPE,Winkler,coverage
seasonal naive,120.404,51.088,568.464,1.0


## The protocol (shared with every notebook)

In [5]:
import itertools
import os
import time
import logging
os.environ["NIXTLA_ID_AS_COL"] = "true"
for name in ("transformers", "root", "cmdstanpy", "prophet", "lightning", "lightning.pytorch", "gluonts",
             "nixtla.nixtla_client", "pytorch_lightning", "neuralforecast"):
    logging.getLogger(name).setLevel(logging.ERROR)

from IPython.display import display
from scipy import signal, stats
from scipy.special import boxcox
from coreforecast.scalers import boxcox_lambda
from statsmodels.tsa.seasonal import MSTL as MSTL_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsforecast.arima import ARIMASummary, ndiffs, nsdiffs
from statsforecast.models import ARIMA, AutoARIMA, AutoETS, AutoTheta, MSTL
try:
    from dotenv import load_dotenv
    load_dotenv()                  # .env next to the notebook holds NIXTLA_API_KEY: never paste the key here
except ImportError:
    pass

dates = data["ds"].reset_index(drop=True)
FREQ = "h"
DAY = 24
WINDOWS = 20                       # rolling 48-hour windows: the SAME on the Mac and in Colab (merged later)
STEP = 24                          # hours between window origins: one origin every midnight (6-25 October)
PART = "day2"                      # the hours a choice is judged on: "day2" (hours 25-48, the private part) or "all"
LAMBDA = 0.45                      # Box-Cox lambda (Guerrero on the last 12 weeks, Section 2.2)
FM_CONTEXT = 2048                  # hours of history a foundation model reads (default)

T0 = time.time()
def elapsed():
    print(f"[{(time.time() - T0) / 60:.1f} min since the start of Section 2]")

In [6]:
NAN = lambda h: np.full(h, np.nan)
FORECASTERS, MIXES, POST, FAMILY, FINAL = {}, {}, {}, {}, {}
bt = None
BASE_COLS = ("window", "step", "actual", "daylight", "benchmark", "benchmark lo", "benchmark hi")
PURGE = -(-H // STEP) - 1          # neighbours this close share hours with a window (1 for 48-hour windows a day apart)


def fwd(x, kind):
    x = np.asarray(x, float)
    if kind == "log":
        return np.log(x)
    if kind == "bc":
        return boxcox(x, LAMBDA)
    return x


def inv(z, kind):
    z = np.asarray(z, float)
    if kind == "log":
        return np.exp(z)
    if kind == "bc":
        return np.power(np.maximum(LAMBDA * z + 1, 1e-12), 1 / LAMBDA)
    return z


def weather_features(df: pd.DataFrame) -> pd.DataFrame:
    # the weather as regressors, plus the physical interactions trees and linear models cannot find alone
    df = df.reset_index(drop=True)
    out = df[["daylight", "cloud_cover", "temperature", "wind_speed", "humidity", "pressure", "air_density"]].astype(float).copy()
    out["sun"] = out["daylight"] * (1 - out["cloud_cover"])         # clear sky while the sun is up
    out["humid_day"] = out["daylight"] * out["humidity"]             # humidity matters while the sun is up
    out["wind3"] = out["wind_speed"] ** 3 / 100                      # wind power grows with the cube of the speed
    # Where the sun is, from the daylight column alone (no outside data): how long the day is, how far into its
    # daylight hours an hour lies, a clear-sky curve peaking at solar noon, and the season. 0/1 daylight cannot say
    # that late-October days are shorter and the sun lower, which is what the models missed in window 1.
    ds = pd.to_datetime(df["ds"])
    day = ds.dt.normalize().to_numpy()
    light = out["daylight"].to_numpy()
    length = pd.Series(light).groupby(day).transform("sum").to_numpy()
    order = pd.Series(light).groupby(day).cumsum().to_numpy()
    position = np.where(light == 1, (order - 0.5) / np.maximum(length, 1), 0.0)
    out["day_length"] = length
    out["sun_position"] = position
    out["clear_sky"] = np.where(light == 1, np.sin(np.pi * position), 0.0)
    out["doy_sin"] = np.sin(2 * np.pi * ds.dt.dayofyear.to_numpy() / 365.25)
    out["doy_cos"] = np.cos(2 * np.pi * ds.dt.dayofyear.to_numpy() / 365.25)
    return out


FEAT_DF = weather_features(pd.concat([data.drop(columns="y"), weather_fc], ignore_index=True))   # history, then the forecast
FEAT_COLS = list(FEAT_DF.columns)
FEAT = FEAT_DF.to_numpy()
ALL_DATES = pd.concat([dates, FUTURE], ignore_index=True)
COV_COLS = ["humidity", "wind_speed", "cloud_cover", "temperature", "daylight", "humid_day", "wind3"]   # the weather
SUN_COLS = ["day_length", "sun_position", "clear_sky", "doy_sin", "doy_cos"]                           # the sun


def cov(cols, start, stop):
    # covariate rows (len(cols), stop - start) for the absolute hours start .. stop-1 (the forecast after the history)
    return FEAT_DF[cols].to_numpy(float)[start:stop].T


def _clean(out, h) -> dict:
    get = lambda *keys: next((np.asarray(out[k], float).ravel()[:h] for k in keys if k in out), NAN(h))
    return {"mean": get("mean"), "lo": get("lo", "lo-95"), "hi": get("hi", "hi-95")}


def on_scale(fit_predict, transform="log"):
    # fit_predict(x, end, h) sees fwd(y[:end]); its forecasts come back on the original scale
    def forecast(end, h):
        out = _clean(fit_predict(fwd(y[:end], transform), end, h), h)
        out = {k: inv(v, transform) for k, v in out.items()}
        out["mean"], out["lo"] = np.maximum(out["mean"], 0), np.maximum(out["lo"], 0)
        return out
    return forecast


def sf(make, transform="log", context=1344, level=True):
    # a statsforecast model: make() builds a fresh one, fitted on the last `context` hours
    def fit_predict(x, end, h):
        x = x[-context:] if context else x
        return make().forecast(y=x, h=h, level=[95]) if level else make().forecast(y=x, h=h)
    return on_scale(fit_predict, transform)


FORECASTERS["benchmark"] = lambda end, h: dict(zip(("mean", "lo", "hi"), snaive(y[:end], h)))


def window_ends(windows=None):
    # where each window ends: window 1 ends with the history, the others STEP hours apart before it
    return [len(y) - (k - 1) * STEP for k in range(windows or WINDOWS, 0, -1)]


def backtest(forecasters: dict, windows=None) -> pd.DataFrame:
    # every forecaster on every window, each window forecast from the hours before it only
    windows = windows or WINDOWS
    ends = window_ends(windows)
    frames = []
    for k, e in zip(range(windows, 0, -1), ends):
        p, lo, hi = snaive(y[:e - H])
        frames.append(pd.DataFrame({"window": k, "step": np.arange(1, H + 1), "actual": y[e - H:e],
                                    "daylight": data["daylight"].to_numpy()[e - H:e],
                                    "benchmark": p, "benchmark lo": lo, "benchmark hi": hi}))
    frame = pd.concat(frames, ignore_index=True)
    bench_rmse = np.mean([rmse(g["actual"], g["benchmark"]) for _, g in frame.groupby("window")])
    seconds = {}
    for name, f in forecasters.items():
        start = time.time()
        try:
            parts = [_clean(f(e - H, H), H) for e in ends]
            if any(np.isnan(p["mean"]).any() for p in parts):
                raise ValueError("NaN in the forecast")
        except Exception as err:
            print(f"  ✗ {name}: {type(err).__name__}: {str(err)[:200]}")
            continue
        FORECASTERS[name] = f
        for key, col in (("mean", name), ("lo", f"{name} lo"), ("hi", f"{name} hi")):
            frame[col] = np.concatenate([p[key] for p in parts])
        seconds[name] = time.time() - start
        r = np.mean([rmse(g["actual"], g[name]) for _, g in frame.groupby("window")])
        print(f"  ✓ {name:<46s} RMSE {r:8.1f}  (skill {r / bench_rmse:.3f})   {seconds[name]:.1f}s", flush=True)
    frame.attrs["seconds"] = seconds
    return frame


def add(bt, new):
    out = bt.drop(columns=[c for c in new.columns if c in bt.columns and c not in BASE_COLS])
    out = pd.concat([out, new[[c for c in new.columns if c not in out.columns]]], axis=1)
    out.attrs["seconds"] = {**bt.attrs.get("seconds", {}), **new.attrs.get("seconds", {})}
    return out


def forecasts_in(bt) -> list[str]:
    return [c for c in bt.columns if c not in BASE_COLS and not c.endswith((" lo", " hi"))] + ["benchmark"]


def _part(frame, part=None):
    # the rows of the judged hours: day 2 = steps 25-48 (the private part), day 1 = steps 1-24, all = 1-48
    part = part or PART
    if part == "day2":
        return frame[frame["step"] > DAY]
    if part == "day1":
        return frame[frame["step"] <= DAY]
    return frame


def _per_window(rows, f, lo=None, hi=None):
    # RMSE, MAPE (and Winkler, coverage) of forecast f in each window of `rows`, as arrays (fast: no groupby)
    _, ix = np.unique(rows["window"].to_numpy(), return_inverse=True)
    n = np.bincount(ix)
    a, f = rows["actual"].to_numpy(float), np.asarray(f, float)
    out = [np.sqrt(np.bincount(ix, (a - f) ** 2) / n), np.bincount(ix, np.abs((a - f) / a)) / n * 100]
    if lo is not None:
        lo, hi = np.asarray(lo, float), np.asarray(hi, float)
        miss = np.clip(lo - a, 0, None) + np.clip(a - hi, 0, None)
        out += [np.bincount(ix, (hi - lo) + 40 * miss) / n, np.bincount(ix, ((a >= lo) & (a <= hi)).astype(float)) / n]
    return out


def window_table(columns=None, per_window=False, part=None) -> pd.DataFrame:
    # mean over the windows of the template's scores on the judged hours; skill = score / benchmark's;
    # won = windows with the lowest RMSE; beat bench = windows with a lower RMSE than the seasonal naive
    columns = list(dict.fromkeys((forecasts_in(bt) if columns is None else list(columns)) + ["benchmark"]))
    rows = _part(bt, part)
    ids = np.unique(rows["window"].to_numpy())
    stats_, per = {}, {}
    for c in columns:
        has = rows[f"{c} lo"].notna().all()
        r, m, *wc = _per_window(rows, rows[c], rows[f"{c} lo"] if has else None, rows[f"{c} hi"] if has else None)
        stats_[c] = {"RMSE": r.mean(), "MAPE": m.mean(), "Winkler": wc[0].mean() if has else np.nan,
                     "coverage": wc[1].mean() if has else np.nan}
        per[c] = pd.Series(r, index=ids)
    tab = pd.DataFrame(stats_).T
    b = tab.loc["benchmark"]
    tab["RMSE skill"], tab["MAPE skill"] = tab["RMSE"] / b["RMSE"], tab["MAPE"] / b["MAPE"]
    tab["point skill"] = (tab["RMSE skill"] + tab["MAPE skill"]) / 2
    tab["Winkler skill"] = tab["Winkler"] / b["Winkler"]
    w_rmse = pd.DataFrame(per)
    tab["won"] = [int((w_rmse.idxmin(axis=1) == c).sum()) for c in tab.index]
    tab["beat bench"] = [int((w_rmse[c] < w_rmse["benchmark"]).sum()) for c in tab.index]
    tab["seconds"] = [bt.attrs.get("seconds", {}).get(c, np.nan) for c in tab.index]
    tab = tab.sort_values("point skill")
    if per_window:
        return tab, (w_rmse.T / w_rmse["benchmark"]).loc[tab.index].round(2)
    return tab


def run(family: str, forecasters: dict, rerun=False) -> pd.DataFrame:
    global bt
    todo = {n: f for n, f in forecasters.items() if rerun or bt is None or n not in bt.columns}
    if todo:
        print(f"{family}: {len(todo)} forecaster(s) x {WINDOWS} windows")
        new = backtest(todo)
        bt = new if bt is None else add(bt, new)
    for n in forecasters:
        if n in bt.columns:
            FAMILY[n] = family
            if n in todo:
                FINAL.pop(n, None)            # re-run: its 48-hour forecast is computed again at the export
    return window_table([n for n in forecasters if n in bt.columns])


def best_of(family: str) -> str:
    names = [n for n, f in FAMILY.items() if f == family and n in bt.columns]
    return window_table(names).drop(index="benchmark")["point skill"].idxmin()


def combine(names, label, how="mean", interval_from=None):
    # the members' mean (or median); the interval: the members' mean interval, or -- interval_from -- that member's
    # own interval stretched to contain the combined point (for members without an interval of their own)
    names = [n for n in dict.fromkeys(names) if n in bt.columns]
    MIXES[label], FAMILY[label] = (names, how, interval_from), "combination"
    FINAL.pop(label, None)
    agg = (lambda cols, **kw: bt[cols].median(axis=1, **kw)) if how == "median" else (lambda cols, **kw: bt[cols].mean(axis=1, **kw))
    bt[label] = agg(names)
    if interval_from:
        bt[f"{label} lo"] = np.minimum(bt[f"{interval_from} lo"], bt[label])
        bt[f"{label} hi"] = np.maximum(bt[f"{interval_from} hi"], bt[label])
    else:
        bt[f"{label} lo"] = agg([f"{n} lo" for n in names], skipna=False)
        bt[f"{label} hi"] = agg([f"{n} hi" for n in names], skipna=False)


def others(w):
    # the windows that share no hour with window w: w and its PURGE neighbours on each side are left out
    return bt[(bt["window"] - w).abs() > PURGE]


def _radius(actual, point, daylight, recipe):
    # the 95% conformal radius from past errors: additive, or additive split day / night
    err = np.abs(np.asarray(actual, float) - np.asarray(point, float))
    if recipe == "conformal day/night":
        return {d: np.quantile(err[np.asarray(daylight) == d], 0.95) for d in (0, 1)}
    return np.quantile(err, 0.95)


def _bounds(point, daylight, r, recipe):
    point = np.asarray(point, float)
    rr = np.where(np.asarray(daylight) == 1, r[1], r[0]) if recipe == "conformal day/night" else r
    return np.maximum(point - rr, 0), point + rr


SCALES = np.round(np.arange(0.5, 1.501, 0.05), 2)


def _scaled(point, lo, hi, c):
    # the interval stretched (c > 1) or shrunk (c < 1) around the point forecast
    point, lo, hi = (np.asarray(v, float) for v in (point, lo, hi))
    return np.maximum(point - c * (point - lo), 0), point + c * (hi - point)


def _best_scale(rows, label):
    # the width factor with the lowest mean Winkler on `rows`
    p, lo, hi = rows[label], rows[f"{label} lo"], rows[f"{label} hi"]
    return min(SCALES, key=lambda c: _per_window(rows, p, *_scaled(p, lo, hi, c))[2].mean())


RECIPES = ["conformal", "conformal day/night"]


def interval_check(label: str, part=None) -> pd.DataFrame:
    # every recipe judged on windows it did not see: radius / width factor from the windows sharing no hour with it
    rows = []
    for w in sorted(bt["window"].unique()):
        g, o = _part(bt[bt["window"] == w], part), _part(others(w), part)
        recipes = {rec: _bounds(g[label], g["daylight"], _radius(o["actual"], o[label], o["daylight"], rec), rec) for rec in RECIPES}
        if g[f"{label} lo"].notna().all():
            recipes["own interval"] = (g[f"{label} lo"].to_numpy(), g[f"{label} hi"].to_numpy())
            recipes["own interval x c"] = _scaled(g[label], g[f"{label} lo"], g[f"{label} hi"], _best_scale(o, label))
        for rec, (lo, hi) in recipes.items():
            s = score(g["actual"], g[label], lo, hi)
            rows.append({"recipe": rec, "window": w, "Winkler": s["Winkler"], "coverage": s["coverage"],
                         "width": float(np.mean(np.asarray(hi) - np.asarray(lo)))})
    return pd.DataFrame(rows).groupby("recipe")[["Winkler", "coverage", "width"]].mean().sort_values("Winkler")


SHADE_FACTORS = np.round(np.arange(0.1, 1.001, 0.05), 2)
SHADE_LEVELS = (50, 100, 150, 200, 300)


def _shade(point, a, level):
    # the night rule: forecasts below `level` multiplied by a (MAPE punishes over-forecasting small values hardest)
    point = np.asarray(point, float)
    return np.where(point < level, a * point, point)


def _tune_post(frame, base, factors=SHADE_FACTORS):
    # the night rule (a, level) with the lowest RMSE skill + MAPE skill on the judged hours of `frame`
    # (base: a Series on frame's index); returns a, level and the RMSE skill + MAPE skill reached
    rows = _part(frame)
    b = base[rows.index].to_numpy()
    br, bm = _per_window(rows, rows["benchmark"])
    obj = lambda a, level: sum(v.mean() / ref.mean() for v, ref in zip(_per_window(rows, _shade(b, a, level)), (br, bm)))
    a, level = min(((a, L) for a in factors for L in SHADE_LEVELS), key=lambda p: obj(*p))
    return float(a), int(level), float(obj(a, level))


def postprocess(day_label, night_label=None, name=None, shade=True) -> str:
    # 1) the day model in daylight hours; at night the same model or its log-scale version, whichever scores
    #    better (night_label fixes it);
    # 2) the night rule: forecasts below `level` multiplied by a (shade=False: no rule).
    # The interval: the one of the model used at each hour, stretched to include the point; its width is judged in
    # interval_check like any model's. The night model, a and level are tuned on the windows sharing no hour with
    # the scored one (purged leave-one-window-out), so bt holds an honest score; the submission uses them tuned on
    # all windows.
    log = f"{day_label} log"
    nights = [night_label] if night_label else [day_label] + ([log] if log in bt.columns else [])
    name = name or f"{day_label} [{'post' if shade else 'hybrid'}]"
    factors = SHADE_FACTORS if shade else (1.0,)
    night = bt["daylight"].to_numpy() == 0
    pick = lambda n, s: pd.Series(np.where(night, bt[n + s], bt[day_label + s]), index=bt.index)

    def tune(frame):
        # [night model, a, level]: the night model whose tuned rule reaches the lowest RMSE skill + MAPE skill
        fits = {n: _tune_post(frame, pick(n, "")[frame.index], factors) for n in nights}
        n = min(fits, key=lambda k: fits[k][2])
        return [n, *fits[n][:2]]

    point, plo, phi = (pd.Series(np.nan, index=bt.index) for _ in range(3))
    chosen = []
    for w in sorted(bt["window"].unique()):
        n, a, level = tune(others(w))
        chosen.append(("log" if n == log else "same" if n == day_label else n, a, level))
        idx = bt.index[bt["window"] == w]
        point[idx] = _shade(pick(n, "")[idx], a, level)
        plo[idx] = np.minimum(pick(n, " lo")[idx], point[idx])
        phi[idx] = np.maximum(pick(n, " hi")[idx], point[idx])
    bt[name], bt[f"{name} lo"], bt[f"{name} hi"] = point, plo, phi
    n, a, level = tune(bt)
    POST[name], FAMILY[name] = {"day": day_label, "night": n, "a": a, "level": level}, "post-processed"
    FINAL.pop(name, None)
    print(f"{name}: night = {n}; submission rule a = {a}, level = {level}; held-out windows chose (night, a, level) "
          f"{pd.Series([str(x) for x in chosen]).value_counts().to_dict()}")
    return name


def from_all_of_y(label: str) -> dict:
    # every forecaster inside `label` once more on the WHOLE history (cached), averaged / post-processed as in bt
    if label not in FINAL:
        if label in MIXES:
            names, how, interval_from = MIXES[label]
            parts = {n: from_all_of_y(n) for n in names}
            agg = np.median if how == "median" else np.mean
            point = agg([p["mean"] for p in parts.values()], axis=0)
            if interval_from:
                lo, hi = np.minimum(parts[interval_from]["lo"], point), np.maximum(parts[interval_from]["hi"], point)
            else:
                lo, hi = (agg([p[k] for p in parts.values()], axis=0) for k in ("lo", "hi"))
            FINAL[label] = {"mean": point, "lo": lo, "hi": hi}
        elif label in POST:
            p = POST[label]
            d_, n_ = from_all_of_y(p["day"]), from_all_of_y(p["night"])
            night = weather_fc["daylight"].to_numpy()[:H] == 0
            base, lo, hi = (np.where(night, n_[k], d_[k]) for k in ("mean", "lo", "hi"))
            point = _shade(base, p["a"], p["level"])
            FINAL[label] = {"mean": point, "lo": np.minimum(lo, point), "hi": np.maximum(hi, point)}
        else:
            FINAL[label] = _clean(FORECASTERS[label](len(y), H), H)
    return FINAL[label]


def final_forecast(label: str, interval: str):
    # the 48 hours from the whole history; a conformal radius or width factor is fitted for each day of the horizon
    # separately (on all windows' day-1 errors for hours 1-24, on their day-2 errors for hours 25-48)
    fc = from_all_of_y(label)
    point = np.maximum(fc["mean"], 0)
    lo, hi = np.array(fc["lo"], float), np.array(fc["hi"], float)
    light = weather_fc["daylight"].to_numpy()[:H]
    for part, sl in (("day1", slice(0, DAY)), ("day2", slice(DAY, H))):
        rows = _part(bt, part)
        if interval == "own interval x c":
            lo[sl], hi[sl] = _scaled(point[sl], fc["lo"][sl], fc["hi"][sl], _best_scale(rows, label))
        elif interval in RECIPES:
            r = _radius(rows["actual"], rows[label], rows["daylight"], interval)
            lo[sl], hi[sl] = _bounds(point[sl], light[sl], r, interval)
        elif interval != "own interval":
            raise ValueError("interval: 'own interval', 'own interval x c' or one of RECIPES")
    if np.isnan(lo).any():
        raise ValueError(f"{label} has no interval of its own: use a conformal recipe")
    return point, np.minimum(np.maximum(lo, 0), point), np.maximum(hi, point)


def plot_validation(names, window=1, show=72):
    g = bt[bt["window"] == window]
    end = len(y) - (window - 1) * STEP
    x = dates.iloc[end - H:end]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(dates.iloc[end - H - show:end - H], y[end - H - show:end - H], color="grey", lw=0.8, label="history")
    ax.plot(x, g["actual"], color="black", lw=1.6, label="actual")
    ax.plot(x, g["benchmark"], color="grey", ls="--", lw=1.2, label="seasonal naive")
    ax.axvline(x.iloc[DAY - 1], color="grey", ls=":", lw=1)
    for n in names:
        ax.plot(x, g[n], lw=1.4, label=f"{n} (RMSE {rmse(g['actual'], g[n]):.1f})")
        if len(names) == 1 and g[f"{n} lo"].notna().all():
            ax.fill_between(x, g[f"{n} lo"], g[f"{n} hi"], alpha=0.2, label="95% interval")
    ax.set_title(f"window {window}" + (" (the template's validation window)" if window == 1 else "") + ": day 1 | day 2")
    ax.legend(loc="upper left", fontsize=8)
    plt.show()


RESULTS_DIR = "results"   # the exported model tables (<prefix>_bt.csv, <prefix>_final.csv) live here


def export(prefix: str):
    # the rolling-window table and each model's 48-hour forecast from the whole history, for merging elsewhere
    names = [n for n in forecasts_in(bt) if n != "benchmark" and n not in MIXES and n not in POST]
    Path(RESULTS_DIR).mkdir(exist_ok=True)
    bt[list(BASE_COLS) + [c for n in names for c in (n, f"{n} lo", f"{n} hi")]].to_csv(f"{RESULTS_DIR}/{prefix}_bt.csv", index=False)
    rows = []
    for n in names:
        try:
            fc = from_all_of_y(n)
        except Exception as err:
            print(f"  ✗ final forecast of {n}: {type(err).__name__}: {str(err)[:120]}")
            continue
        rows += [{"model": n, "family": FAMILY.get(n, "?"), "step": i + 1, "mean": fc["mean"][i], "lo": fc["lo"][i],
                  "hi": fc["hi"][i]} for i in range(H)]
    pd.DataFrame(rows).to_csv(f"{RESULTS_DIR}/{prefix}_final.csv", index=False)
    print(f"saved {RESULTS_DIR}/{prefix}_bt.csv ({len(names)} models) and {RESULTS_DIR}/{prefix}_final.csv")


def merge(prefix: str):
    # adds another notebook's exported models to bt, FAMILY and FINAL (the same windows are checked)
    global bt
    other = pd.read_csv(f"{RESULTS_DIR}/{prefix}_bt.csv")
    assert len(other) == len(bt) and (other["window"].to_numpy() == bt["window"].to_numpy()).all() \
        and np.allclose(other["actual"], bt["actual"]), "not the same windows: WINDOWS and STEP must match"
    new = [c for c in other.columns if c not in BASE_COLS]
    seconds = bt.attrs.get("seconds", {})
    bt = pd.concat([bt.drop(columns=[c for c in new if c in bt.columns]), other[new]], axis=1)
    bt.attrs["seconds"] = seconds
    fin = pd.read_csv(f"{RESULTS_DIR}/{prefix}_final.csv")
    for n, g in fin.groupby("model"):
        g = g.sort_values("step")
        FINAL[n] = {k: g[k].to_numpy(float) for k in ("mean", "lo", "hi")}
        FAMILY[n] = g["family"].iloc[0]
    print(f"merged {fin['model'].nunique()} models from {prefix}")

In [7]:
# Helpers used by the family notebooks
import os

RESUME = True       # <-- True: the models already in this notebook's export (an earlier run) are read back, only new
                    #     sections compute; False: everything from scratch
CONTEXTS = {"1w": 168, "2w": 336, "4w": 672, "8w": 1344, "12w": 2016, "26w": 4368}   # training lengths (hours)
LENGTH_ORDER = ["3d", "5d", "1w", "10d", "2w", "3w", "4w", "8w", "12w", "26w",
                "14d", "28d", "56d", "60d", "90d", "120d", "180d", "365d", "730d", "1095d"]


def resume(prefix):
    # RESUME: read back the models of this notebook's earlier export, so run() computes only the new ones
    global bt
    if RESUME and os.path.exists(f"{RESULTS_DIR}/{prefix}_bt.csv"):
        if bt is None:
            bt = pd.read_csv(f"{RESULTS_DIR}/{prefix}_bt.csv")[list(BASE_COLS)]
        merge(prefix)


def sf_any(make, transform="log", context=1344):
    # a statsforecast model with its 95% interval when it has one, the point forecast alone when it has not
    def fit_predict(x, end, h):
        x = x[-context:] if context else x
        try:
            return make().forecast(y=x, h=h, level=[95])
        except Exception:
            return make().forecast(y=x, h=h)
    return on_scale(fit_predict, transform)


def grid_view(table, value="point skill"):
    # "name, training length" rows -> a model x training-length table
    t = table.drop(index="benchmark").copy()
    t = t[[", " in n for n in t.index]]
    t["model"] = [n.rsplit(", ", 1)[0] for n in t.index]
    t["training"] = [n.rsplit(", ", 1)[1] for n in t.index]
    order = [c for c in LENGTH_ORDER if c in set(t["training"])]
    return t.pivot(index="model", columns="training", values=value)[order]


def show(table, title):
    print(title)
    display(table[["RMSE skill", "MAPE skill", "point skill", "Winkler skill", "coverage", "beat bench", "seconds"]].head(15).round(3))


# Regressors of the linear models: the weather + the sun (the set of the best DHR). Cloud-based interactions
# (clear-sky curve x (1 - cloud)) were tried and made linear models worse: cloud cover acts non-linearly here (EDA).
X_REG = ["humidity", "wind_speed", "cloud_cover", "temperature", "daylight", "humid_day", "wind3", "day_length", "clear_sky"]
X_DAILY = ["humid_day", "wind3", "temperature", "day_length"]          # daily means of these: a day's weather


def _try_forecast(model, y_, h, X=None, X_future=None):
    # the model's 95% interval when it has one, the point forecast alone when it has not
    kw = {} if X is None else {"X": X, "X_future": X_future}
    try:
        return model.forecast(y=y_, h=h, level=[95], **kw)
    except Exception:
        return model.forecast(y=y_, h=h, **kw)


def _varying(X, Xf):
    # drops the columns constant on the training rows (a regression cannot use them; e.g. daylight at 3 a.m.)
    keep = np.asarray(X).std(axis=0) > 1e-9
    return (X[:, keep], Xf[:, keep]) if keep.any() else (None, None)


def by_hour(make, transform="log", days=28, xcols=None):
    # one series per hour of the day: the last `days` values of each of the 24 hours form a daily series, forecast
    # 1 and 2 days ahead by make(); with xcols, that hour's weather as regressors (models that take X)
    def fit_predict(x, end, h):
        n = min(days, end // DAY)
        t = np.arange(end - n * DAY, end).reshape(n, DAY)                # rows: days; columns: the hours of the day
        steps = -(-h // DAY)
        tf = np.arange(end, end + steps * DAY).reshape(steps, DAY)
        F = FEAT_DF[xcols].to_numpy(float) if xcols else None
        out = {k: np.full(steps * DAY, np.nan) for k in ("mean", "lo", "hi")}
        for p in range(DAY):
            X, Xf = _varying(F[t[:, p]], F[tf[:, p]]) if xcols else (None, None)
            fc = _try_forecast(make(), np.asarray(x[t[:, p]], float), steps, X, Xf)
            for k, key in (("mean", "mean"), ("lo", "lo-95"), ("hi", "hi-95")):
                if key in fc:
                    out[k][p::DAY] = np.asarray(fc[key], float)[:steps]
        return {k: v[:h] for k, v in out.items()}
    return on_scale(fit_predict, transform)


def top_down(make, days=56, profile_days=7, transform="log", xcols=None):
    # the day's total (sum of its 24 hours) forecast by make() from the last `days` daily totals (on `transform`),
    # spread over the hours with the mean profile -- each hour's share of its day -- of the last `profile_days` days;
    # with xcols, the daily means of those columns as regressors (the forecast days' weather is known)
    def fit_predict(x, end, h):                                           # x: the original scale
        n = min(days, end // DAY)
        t = np.arange(end - n * DAY, end)
        block = np.asarray(x[t], float).reshape(n, DAY)
        recent = block[-profile_days:]
        share = (recent / recent.sum(axis=1, keepdims=True)).mean(axis=0)
        steps = -(-h // DAY)
        X = Xf = None
        if xcols:
            F = FEAT_DF[xcols].to_numpy(float)
            X = F[t].reshape(n, DAY, -1).mean(axis=1)
            Xf = F[np.arange(end, end + steps * DAY)].reshape(steps, DAY, -1).mean(axis=1)
            X, Xf = _varying(X, Xf)
        fc = _try_forecast(make(), fwd(block.sum(axis=1), transform), steps, X, Xf)
        return {k: np.concatenate([inv(np.asarray(fc[key], float)[d], transform) * share for d in range(steps)])[:h]
                for k, key in (("mean", "mean"), ("lo", "lo-95"), ("hi", "hi-95")) if key in fc}
    return on_scale(fit_predict, None)


def two_stage(make, xcols=X_REG, transform="log", context=2016, alpha=30.0):
    # regression with ETS (or other) errors in two stages: (1) ridge regression of the transformed series on the hour
    # of the day (24 dummies, not penalised) and the standardised weather, on the training window; (2) make() -- a
    # non-seasonal model -- on the DAILY MEANS of its residuals (what the weather does not explain: a level that moves),
    # forecast 1 and 2 days ahead and added to every hour of those days. (On the hourly residuals the last night's
    # residual was carried into tomorrow's midday and multiplied the peak.)
    def fit_predict(x, end, h):
        n = min(context, end) // DAY * DAY
        t, tf = np.arange(end - n, end), np.arange(end, end + h)
        hours = ALL_DATES.dt.hour.to_numpy()
        F = FEAT_DF[xcols].to_numpy(float)
        mu, sd = F[t].mean(axis=0), F[t].std(axis=0) + 1e-9
        design = lambda r: np.column_stack([(hours[r][:, None] == np.arange(DAY)).astype(float), (F[r] - mu) / sd])
        X, Xf = design(t), design(tf)
        z = np.asarray(x[t], float)
        beta = np.linalg.solve(X.T @ X + np.diag(np.r_[np.zeros(DAY), np.full(len(xcols), alpha)]), X.T @ z)
        resid = z - X @ beta
        steps = -(-h // DAY)
        fc = _try_forecast(make(), resid.reshape(-1, DAY).mean(axis=1), steps)
        reg = Xf @ beta
        spread = np.quantile(np.abs(resid - np.repeat(resid.reshape(-1, DAY).mean(axis=1), DAY)), 0.95)
        out = {"mean": reg + np.repeat(np.asarray(fc["mean"], float), DAY)[:h]}
        if "lo-95" in fc:   # the daily level's interval widened by the hours' spread around their day's mean
            out["lo"] = reg + np.repeat(np.asarray(fc["lo-95"], float), DAY)[:h] - spread
            out["hi"] = reg + np.repeat(np.asarray(fc["hi-95"], float), DAY)[:h] + spread
        return out
    return on_scale(fit_predict, transform)


def sf_x(make, xcols, transform="log", context=1344):
    # a statsforecast model that takes regressors (X, X_future): the weather of the training hours and of the horizon
    def fit_predict(x, end, h):
        t0 = max(0, end - context)
        F = FEAT_DF[xcols].to_numpy(float)
        return _try_forecast(make(), np.asarray(x[t0:], float), h, F[t0:end], F[end:end + h])
    return on_scale(fit_predict, transform)

In [8]:
resume("family_ml")      # reads back an earlier run when RESUME = True (see the cell above)

merged 17 models from family_ml


## 1. One design for every regression model

Every hour of the horizon becomes a row: hour of day, the weather and sun features of that hour (actual weather on
the windows, the forecast for the submission), lags that are known at the forecast origin (≥ 24 hours for day 1,
≥ 48 for day 2: **two stages**, one model per day), and the recent level. The target is log y **minus the recent
level** (the mean of the last 7 days of log y), so a level never seen before (the 2026 growth) does not confuse the
models. Trained on the `days` days before each window.

In [9]:
def design(x, rows, min_lag, relative=True):
    # features of the hours in `rows`, using only values at least `min_lag` hours old; and the level to add back
    xs = pd.Series(np.r_[x, np.full(H, np.nan)])
    dd = ALL_DATES.iloc[rows]
    level = xs.shift(min_lag).rolling(7 * DAY).mean().to_numpy()[rows]
    cols = {"hour": dd.dt.hour.to_numpy()}
    for k in [min_lag + DAY * i for i in range(6)]:
        cols[f"lag{k}"] = xs.shift(k).to_numpy()[rows] - (level if relative else 0)
    cols["mean_last_day"] = xs.shift(min_lag).rolling(DAY).mean().to_numpy()[rows] - (level if relative else 0)
    cols["level_7d"] = level
    cols.update({c: FEAT_DF[c].to_numpy()[rows] for c in FEAT_COLS})
    return pd.DataFrame(cols), level


def _row_weights(rows, end, kind):
    # training-row weights: "recent" halves every 180 days back; "season" favours the same weeks of other years
    # (a Gaussian of +-30 days around the forecast date's day of the year, plus a small floor)
    if not kind:
        return None
    w = np.ones(len(rows))
    if "recent" in kind:
        w *= 0.5 ** ((end - rows) / DAY / 180)
    if "season" in kind:
        doy = ALL_DATES.dt.dayofyear.to_numpy()
        gap = np.abs(doy[rows] - doy[end])
        w *= np.exp(-0.5 * (np.minimum(gap, 365 - gap) / 30) ** 2) + 0.05
    return w


def regression_model(make, days=365, transform="log", relative=True, hour_onehot=False, quantile_fn=None, weight=None):
    # make() builds a fresh scikit-learn-style regressor; two stages (day 1: lags >= 24, day 2: lags >= 48);
    # weight: None, "recent", "season" or "season + recent" (models whose fit takes sample_weight)
    def fit_predict(x, end, h):
        rows = np.arange(max(400, end - DAY * days), end)
        w = _row_weights(rows, end, weight)
        out = {"mean": np.full(h, np.nan)}
        for min_lag, a, b in ((DAY, 0, DAY), (2 * DAY, DAY, h)):
            X, lev = design(x, rows, min_lag, relative)
            target = x[rows] - (lev if relative else 0)
            ok = ~np.isnan(target) & ~X.isna().any(axis=1).to_numpy()
            Xf, levf = design(x, np.arange(end + a, end + b), min_lag, relative)
            if hour_onehot:
                X, Xf = (pd.get_dummies(m.astype({"hour": "category"}), columns=["hour"], dtype=float) for m in (X, Xf))
                Xf = Xf.reindex(columns=X.columns, fill_value=0.0)
            model = make()
            model.fit(X[ok], target[ok], **({} if w is None else {"sample_weight": w[ok]}))
            out["mean"][a:b] = model.predict(Xf.fillna(0)) + (levf if relative else 0)
        return out
    return on_scale(fit_predict, transform)

## 2. Linear models, trees, gradient boosting

In [10]:
import lightgbm as lgb
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

LINEAR = {
    "Ridge (hour dummies) relative log, 365d": regression_model(lambda: make_pipeline(StandardScaler(), Ridge(alpha=1.0)), 365, hour_onehot=True),
    "Ridge (hour dummies) relative log, 90d":  regression_model(lambda: make_pipeline(StandardScaler(), Ridge(alpha=1.0)), 90, hour_onehot=True),
}
show(run("regression", LINEAR), "linear regression on the same features (day 2):")

TREES = {
    "Random forest relative log, 365d":  regression_model(lambda: RandomForestRegressor(300, min_samples_leaf=5, max_features=0.5, n_jobs=-1, random_state=1), 365),
    "Extra trees relative log, 365d":    regression_model(lambda: ExtraTreesRegressor(300, min_samples_leaf=5, max_features=0.5, n_jobs=-1, random_state=1), 365),
    "HistGB relative log, 365d":         regression_model(lambda: HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, random_state=1), 365),
    "HistGB relative log, 730d":         regression_model(lambda: HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, random_state=1), 730),
}
show(run("trees", TREES), "random forest, extra trees, histogram gradient boosting (day 2):")

LGB = lambda **kw: (lambda: lgb.LGBMRegressor(**{"n_estimators": 600, "learning_rate": 0.03, "num_leaves": 63, "min_child_samples": 20,
                                                 "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8, "verbose": -1,
                                                 "random_state": 1, **kw}))
LGBMS = {
    "LightGBM relative log, 730d":                  regression_model(LGB(), 730),
    "LightGBM relative log, 365d":                  regression_model(LGB(), 365),
    "LightGBM small trees relative log, 730d":      regression_model(LGB(num_leaves=15, n_estimators=1200), 730),
    "LightGBM big trees relative log, 730d":        regression_model(LGB(num_leaves=127, min_child_samples=40), 730),
    "LightGBM relative bc, 730d":                   regression_model(LGB(), 730, transform="bc"),
}
show(run("LightGBM", LGBMS), "LightGBM variants (day 2):")
elapsed()

linear regression on the same features (day 2):


,RMSE skill,MAPE skill,point skill,Winkler skill,coverage,beat bench,seconds
"Ridge (hour dummies) relative log, 90d",1.063,0.293,0.678,NaN,NaN,11,NaN
"Ridge (hour dummies) relative log, 365d",1.310,0.232,0.771,NaN,NaN,4,NaN
benchmark,1.000,1.000,1.000,1.0,0.817,0,NaN


random forest, extra trees, histogram gradient boosting (day 2):


,RMSE skill,MAPE skill,point skill,Winkler skill,coverage,beat bench,seconds
"HistGB relative log, 730d",0.678,0.244,0.461,NaN,NaN,16,NaN
"Extra trees relative log, 365d",0.752,0.206,0.479,NaN,NaN,14,NaN
"HistGB relative log, 365d",0.804,0.205,0.504,NaN,NaN,13,NaN
"Random forest relative log, 365d",0.760,0.318,0.539,NaN,NaN,13,NaN
benchmark,1.000,1.000,1.000,1.0,0.817,0,NaN


LightGBM variants (day 2):


,RMSE skill,MAPE skill,point skill,Winkler skill,coverage,beat bench,seconds
"LightGBM relative log, 730d",0.677,0.207,0.442,NaN,NaN,17,NaN
"LightGBM small trees relative log, 730d",0.723,0.211,0.467,NaN,NaN,15,NaN
"LightGBM big trees relative log, 730d",0.727,0.220,0.474,NaN,NaN,15,NaN
"LightGBM relative log, 365d",0.742,0.252,0.497,NaN,NaN,14,NaN
"LightGBM relative bc, 730d",0.701,0.451,0.576,NaN,NaN,15,NaN
benchmark,1.000,1.000,1.000,1.0,0.817,0,NaN


[0.0 min since the start of Section 2]


## 3. Prophet with the weather and the sun

In [11]:
from prophet import Prophet
logging.getLogger("cmdstanpy").disabled = True


def prophet_model(days=120, xcols=None, transform="log"):
    def fit_predict(x, end, h):
        start = max(0, end - DAY * days)
        df = pd.DataFrame({"ds": dates.iloc[start:end].to_numpy(), "y": x[start:]})
        fut = pd.DataFrame({"ds": ALL_DATES.iloc[end:end + h].to_numpy()})
        model = Prophet(daily_seasonality=10, weekly_seasonality=False, yearly_seasonality=days >= 730,
                        interval_width=0.95, changepoint_prior_scale=0.05, uncertainty_samples=300)
        for c in xcols or []:
            model.add_regressor(c)
            df[c] = FEAT_DF[c].to_numpy()[start:end]
            fut[c] = FEAT_DF[c].to_numpy()[end:end + h]
        model.fit(df)
        fc = model.predict(fut)
        return {"mean": fc["yhat"].to_numpy(), "lo": fc["yhat_lower"].to_numpy(), "hi": fc["yhat_upper"].to_numpy()}
    return on_scale(fit_predict, transform)


W = ["humidity", "wind_speed", "cloud_cover", "temperature", "daylight", "humid_day", "wind3"]
PROPHETS = {
    "Prophet + weather log, 120d":       prophet_model(120, W),
    "Prophet + weather + sun log, 120d": prophet_model(120, W + ["day_length", "clear_sky"]),
    "Prophet + weather + sun log, 60d":  prophet_model(60, W + ["day_length", "clear_sky"]),
}
show(run("Prophet", PROPHETS), "Prophet (day 2):")
elapsed()

Prophet (day 2):


,RMSE skill,MAPE skill,point skill,Winkler skill,coverage,beat bench,seconds
"Prophet + weather + sun log, 120d",0.781,0.306,0.543,0.660,0.808,16,NaN
"Prophet + weather log, 120d",0.796,0.300,0.548,0.672,0.800,15,NaN
"Prophet + weather + sun log, 60d",0.931,0.279,0.605,0.607,0.827,9,NaN
benchmark,1.000,1.000,1.000,1.000,0.817,0,NaN


[0.0 min since the start of Section 2]


## 4. Analog days: the k most similar past hours

A classic of energy forecasting (the *analog ensemble*): for every hour of the horizon, find among the past year's
hours **at the same hour of the day** the `k` with the most similar weather and sun (standardised humidity, cloud,
temperature, wind, clear-sky curve, day length), and forecast with their generation **relative to their own recent
level**, times today's level. The spread of the analogs gives an interval of its own (their 2.5% and 97.5% quantiles).
No fitting, nothing to tune but `k`.

In [12]:
from sklearn.neighbors import NearestNeighbors

ANALOG_X = ["humidity", "cloud_cover", "temperature", "wind_speed", "clear_sky", "day_length"]


def analog_model(k=40, days=365):
    def fit_predict(x, end, h):
        xs = pd.Series(np.r_[x, np.full(H, np.nan)])
        level = xs.shift(2 * DAY).rolling(7 * DAY).mean().to_numpy()          # known 2 days ahead: both days of the horizon
        rows = np.arange(max(8 * DAY, end - DAY * days), end)
        target_rows = np.arange(end, end + h)
        F = FEAT_DF[ANALOG_X].to_numpy(float)
        mu, sd = F[rows].mean(axis=0), F[rows].std(axis=0) + 1e-9
        Z = (F - mu) / sd
        hours = ALL_DATES.dt.hour.to_numpy()
        rel = x[rows] - level[rows]                                          # log y relative to its level
        out = {k_: np.full(h, np.nan) for k_ in ("mean", "lo", "hi")}
        for hr in np.unique(hours[target_rows]):
            past = rows[(hours[rows] == hr) & ~np.isnan(rel)]
            nn = NearestNeighbors(n_neighbors=min(k, len(past))).fit(Z[past])
            tgt = target_rows[hours[target_rows] == hr]
            _, idx = nn.kneighbors(Z[tgt])
            vals = (x[past] - level[past])[idx]                              # (len(tgt), k)
            pos = tgt - end
            out["mean"][pos] = np.median(vals, axis=1) + level[tgt]
            out["lo"][pos] = np.quantile(vals, 0.025, axis=1) + level[tgt]
            out["hi"][pos] = np.quantile(vals, 0.975, axis=1) + level[tgt]
        return out
    return on_scale(fit_predict, "log")


show(run("analog", {f"Analog days k={k}, 365d": analog_model(k) for k in (20, 40, 80)}), "analog days (day 2):")
elapsed()

analog days (day 2):


,RMSE skill,MAPE skill,point skill,Winkler skill,coverage,beat bench,seconds
"Analog days k=80, 365d",1.037,0.520,0.778,0.574,0.856,8,NaN
"Analog days k=40, 365d",1.101,0.461,0.781,0.636,0.827,7,NaN
"Analog days k=20, 365d",1.122,0.520,0.821,0.716,0.777,6,NaN
benchmark,1.000,1.000,1.000,1.000,0.817,0,NaN


[0.0 min since the start of Section 2]


## 5. Which past to learn from: weighting the training rows, and a longer window

LightGBM on 730 days beat 365 days — more examples of how the weather drives the output — but those days include the
summers, when the sun is high and the response to clouds different. Late October is best learned from **the same
weeks of other years** and from **recent weeks**:
- **season-weighted**: rows weighted by a Gaussian of ±30 days around the forecast date's day of the year (plus a small
  floor), so other years' Octobers–Novembers count most;
- **recent-weighted**: weights halve every 180 days back;
- and a **3-year window** (1095 days).

In [13]:
WEIGHTED = {
    "LightGBM season-weighted relative log, 730d": regression_model(LGB(), 730, weight="season"),
    "LightGBM recent-weighted relative log, 730d": regression_model(LGB(), 730, weight="recent"),
    "LightGBM relative log, 1095d":                regression_model(LGB(), 1095),
}
show(run("LightGBM", WEIGHTED), "LightGBM: weights and a longer window (day 2):")
elapsed()

LightGBM: 3 forecaster(s) x 20 windows


  ✓ LightGBM season-weighted relative log, 730d    RMSE    232.0  (skill 0.742)   316.6s


  ✓ LightGBM recent-weighted relative log, 730d    RMSE    234.3  (skill 0.749)   333.8s


  ✓ LightGBM relative log, 1095d                   RMSE    227.4  (skill 0.727)   351.2s


LightGBM: weights and a longer window (day 2):


,RMSE skill,MAPE skill,point skill,Winkler skill,coverage,beat bench,seconds
"LightGBM relative log, 1095d",0.701,0.233,0.467,NaN,NaN,16,351.239
"LightGBM season-weighted relative log, 730d",0.739,0.207,0.473,NaN,NaN,14,316.640
"LightGBM recent-weighted relative log, 730d",0.726,0.244,0.485,NaN,NaN,16,333.773
benchmark,1.000,1.000,1.000,1.0,0.817,0,NaN


[16.7 min since the start of Section 2]


## Summary and export

In [14]:
summary, per_window = window_table(per_window=True)
print("day 2 (the private hours):")
display(summary.head(30))
print("all 48 hours:")
display(window_table(part="all").head(15))
print("day-2 RMSE relative to the seasonal naive, window by window (window 1 = the most recent):")
display(per_window.head(15))
for f in dict.fromkeys(FAMILY.values()):
    if f != "simple":
        print(f"best {f}: {best_of(f)}")
export("family_ml")
elapsed()

day 2 (the private hours):


,RMSE,MAPE,Winkler,coverage,RMSE skill,MAPE skill,point skill,Winkler skill,won,beat bench,seconds
"LightGBM relative log, 730d",227.204,236.128,NaN,NaN,0.677,0.207,0.442,NaN,4,17,NaN
"HistGB relative log, 730d",227.497,278.242,NaN,NaN,0.678,0.244,0.461,NaN,0,16,NaN
"LightGBM relative log, 1095d",235.356,265.599,NaN,NaN,0.701,0.233,0.467,NaN,3,16,351.239
"LightGBM small trees relative log, 730d",242.737,241.373,NaN,NaN,0.723,0.211,0.467,NaN,2,15,NaN
"LightGBM season-weighted relative log, 730d",247.945,236.112,NaN,NaN,0.739,0.207,0.473,NaN,0,14,316.640
"LightGBM big trees relative log, 730d",244.036,251.532,NaN,NaN,0.727,0.220,0.474,NaN,0,15,NaN
"Extra trees relative log, 365d",252.472,235.465,NaN,NaN,0.752,0.206,0.479,NaN,2,14,NaN
"LightGBM recent-weighted relative log, 730d",243.633,278.787,NaN,NaN,0.726,0.244,0.485,NaN,0,16,333.773
"LightGBM relative log, 365d",249.031,287.756,NaN,NaN,0.742,0.252,0.497,NaN,0,14,NaN
"HistGB relative log, 365d",269.832,233.432,NaN,NaN,0.804,0.205,0.504,NaN,0,13,NaN


all 48 hours:


,RMSE,MAPE,Winkler,coverage,RMSE skill,MAPE skill,point skill,Winkler skill,won,beat bench,seconds
"LightGBM relative log, 730d",227.858,175.730,NaN,NaN,0.729,0.233,0.481,NaN,2,17,NaN
"LightGBM season-weighted relative log, 730d",231.998,171.648,NaN,NaN,0.742,0.228,0.485,NaN,2,17,316.640
"HistGB relative log, 730d",227.994,194.558,NaN,NaN,0.729,0.258,0.494,NaN,1,17,NaN
"LightGBM relative log, 1095d",227.422,197.697,NaN,NaN,0.727,0.262,0.495,NaN,1,17,351.239
"LightGBM small trees relative log, 730d",240.865,174.258,NaN,NaN,0.770,0.231,0.501,NaN,1,16,NaN
"LightGBM recent-weighted relative log, 730d",234.320,191.568,NaN,NaN,0.749,0.254,0.502,NaN,0,17,333.773
"LightGBM big trees relative log, 730d",237.049,189.445,NaN,NaN,0.758,0.251,0.505,NaN,0,17,NaN
"LightGBM relative log, 365d",239.451,195.807,NaN,NaN,0.766,0.260,0.513,NaN,0,17,NaN
"Extra trees relative log, 365d",243.973,198.231,NaN,NaN,0.780,0.263,0.522,NaN,0,15,NaN
"HistGB relative log, 365d",263.783,167.071,NaN,NaN,0.843,0.222,0.533,NaN,0,13,NaN


day-2 RMSE relative to the seasonal naive, window by window (window 1 = the most recent):


,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
"LightGBM relative log, 730d",1.51,0.41,0.23,0.63,0.55,0.42,0.38,0.29,0.39,0.97,0.77,0.41,0.58,0.90,0.85,1.30,1.46,0.50,0.94,0.95
"HistGB relative log, 730d",1.53,0.40,0.33,0.64,0.63,0.43,0.38,0.31,0.38,0.90,0.73,0.44,0.59,0.88,0.89,1.34,1.54,0.42,0.88,1.16
"LightGBM relative log, 1095d",1.34,0.36,0.58,0.56,0.60,0.53,0.34,0.41,0.35,0.91,0.73,0.35,0.64,0.84,0.99,1.34,1.39,0.61,0.98,1.14
"LightGBM small trees relative log, 730d",1.68,0.34,0.31,0.64,0.72,0.51,0.55,0.56,0.41,0.93,0.70,0.48,0.46,0.88,1.01,1.12,1.52,0.48,1.08,0.95
"LightGBM season-weighted relative log, 730d",1.49,0.46,0.33,0.70,0.55,0.38,0.39,0.29,0.46,0.95,0.78,0.53,0.93,1.01,1.15,1.60,1.44,0.53,0.97,1.06
"LightGBM big trees relative log, 730d",1.67,0.36,0.24,0.86,0.76,0.46,0.39,0.33,0.44,1.02,0.77,0.42,0.71,0.87,0.88,1.35,1.58,0.56,0.95,1.08
"Extra trees relative log, 365d",1.83,0.32,0.23,0.74,0.57,0.31,0.37,0.39,0.38,0.93,0.76,0.48,0.84,1.09,1.26,1.38,1.61,0.73,0.93,1.02
"LightGBM recent-weighted relative log, 730d",1.64,0.39,0.24,0.64,0.85,0.36,0.35,0.39,0.39,0.92,0.79,0.41,0.71,0.92,0.96,1.43,1.69,0.59,0.91,1.17
"LightGBM relative log, 365d",1.25,0.71,0.29,0.71,0.64,0.31,0.37,0.33,0.51,0.94,0.75,0.40,0.77,0.95,1.30,1.43,1.53,0.67,1.00,1.21
"HistGB relative log, 365d",1.26,0.78,0.31,0.77,0.57,0.38,0.46,0.30,0.58,1.00,0.84,0.42,0.88,1.03,1.46,1.53,1.69,0.73,1.10,1.31


best analog: Analog days k=80, 365d


best trees: HistGB relative log, 730d
best LightGBM: LightGBM relative log, 730d


best Prophet: Prophet + weather + sun log, 120d
best regression: Ridge (hour dummies) relative log, 90d


saved family_ml_bt.csv (20 models) and family_ml_final.csv
[17.6 min since the start of Section 2]
